# Van der Pol — the reference ROA computation

**Goal.** Reproduce the canonical region-of-attraction result for the time-reversed
Van der Pol oscillator and check it against ground truth.

**Why this system.** Its exact ROA boundary is computable: the reversed system's ROA
is bounded by the forward system's stable limit cycle, which a numerical integrator
finds to machine precision. Almost no other nonlinear benchmark gives an inner
approximation something exact to be measured against.

**What a reader should take away:** that the author can set up an SOS program, knows
what the S-procedure is doing, and measures the result against ground truth instead
of against another SOS run.

**Prerequisites:** none. This is the entry point of the repository.

## 0. Setup

Start the Julia backend and load the LyapunovToolbox package. The first call to `get_session()` starts a
persistent Julia process that lives for the duration of the notebook kernel — subsequent calls return the
same session. If the session fails to start, check that Julia ≥ 1.10 is on `PATH` and that `julia/Project.toml`
has been instantiated (`julia --project=julia -e 'import Pkg; Pkg.instantiate()'` from the repo root).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from lyapunov_toolbox import julia_available
from lyapunov_toolbox.julia_bridge import get_session, certify_roa, synthesize
from lyapunov_toolbox.plotting import (
    plot_roa_2d, plot_phase_portrait, plot_vdot_field,
    plot_roa_comparison, plot_vs_convergence, save_figure,
)
from lyapunov_toolbox.plotting import use_style
use_style()

# The first call across the bridge boots Julia and precompiles SumOfSquares.
# That is 30-60 s on a cold cache. Do it here, in its own cell, so the cost is
# visible to a reader rather than mysteriously attached to the first plot.
session = get_session()
print(f"Julia {session.version} ready.")

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


  Activating project at `C:\Users\aliei\Documents\Multi-Agent\lyapunov-toolbox\julia`


Julia 1.12.6 ready.


## 1. The system

$$\dot x_1 = -x_2, \qquad \dot x_2 = x_1 + \mu(x_1^2 - 1)x_2$$

with $\mu = 1$. The time-reversed Van der Pol has an unstable equilibrium at the origin
whose region of attraction is exactly the interior of the *forward* system's stable limit cycle.
This is rare: for almost all nonlinear systems the exact ROA boundary is unknown, but here
we have a computable ground truth.

The code cell below defines the dynamics as Julia strings. The bridge parses them into
`DynamicPolynomials` objects and passes them to the SOS solver. The `reversed=true` flag
negates the RHS to get the time-reversed field; the forward system is available at
`reversed=false` for limit-cycle computation.

In [2]:
# Time-reversed Van der Pol: ẋ₁ = -x₂, ẋ₂ = x₁ + μ(x₁² - 1)x₂
mu = 1.0

def vdp_field(x):
    """Time-reversed Van der Pol vector field."""
    x1, x2 = x[0], x[1]
    return np.array([-x2, x1 + mu * (x1**2 - 1) * x2])

# Jacobian at origin.
A = np.array([[0, -1], [1, -mu]])
evals = np.linalg.eigvals(A)
print(f"A = [[0, -1], [1, -{mu}]]")
print(f"Eigenvalues: {evals}")
assert np.all(np.real(evals) < 0), "Origin is not Hurwitz!"
print("Origin is Hurwitz ✓")

# Phase portrait with limit cycle computed below.
fig, ax = plt.subplots(figsize=(7, 7))
x = np.linspace(-3, 3, 25)
y = np.linspace(-3, 3, 25)
X, Y = np.meshgrid(x, y)
U = -Y
V = X + mu * (X**2 - 1) * Y
speed = np.sqrt(U**2 + V**2)
ax.streamplot(X, Y, U, V, color='grey', linewidth=0.8*speed/(speed.max()+1e-8),
              density=1.5, arrowsize=0.5)
ax.set_xlabel("$x_1$")
ax.set_ylabel("$x_2$")
ax.set_aspect("equal")
ax.set_title(f"Time-reversed Van der Pol (μ = {mu})")
ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
plt.show()

A = [[0, -1], [1, -1.0]]
Eigenvalues: [-0.5+0.8660254j -0.5-0.8660254j]
Origin is Hurwitz ✓


C:\Users\aliei\AppData\Local\Temp\ipykernel_21776\2350416714.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. The Lyapunov candidate from the linearisation

$$A = \left.\frac{\partial f}{\partial x}\right|_{x=0} = \begin{bmatrix} 0 & -1 \\ 1 & -1 \end{bmatrix}$$

Solve $A^\top P + P A = -I$ for the quadratic Lyapunov function $V_2(x) = x^\top P x$.
This is the *starting point* of the V–s iteration — the iteration will refine it to a
higher-degree polynomial that captures more of the true ROA.

In [3]:
# Solve A'P + PA = -I for the linearisation.
from scipy.linalg import solve_continuous_lyapunov
P = solve_continuous_lyapunov(A.T, -np.eye(2))
print(f"P =\n{P}")
# Verify the Lyapunov residual.
residual = A.T @ P + P @ A + np.eye(2)
print(f"‖A'P + PA + I‖ = {np.linalg.norm(residual):.2e}")
assert np.linalg.norm(residual) < 1e-10, "Lyapunov equation residual too large"

# V(x) = x' P x gives a quadratic Lyapunov function near the origin.
def V_quad(x):
    return x @ P @ x

# Plot level curves.
fig, ax = plt.subplots(figsize=(7, 7))
x = np.linspace(-3, 3, 25)
y = np.linspace(-3, 3, 25)
X, Y = np.meshgrid(x, y)
U = -Y
Vf = X + mu * (X**2 - 1) * Y
speed = np.sqrt(U**2 + Vf**2)
ax.streamplot(X, Y, U, Vf, color='grey', linewidth=0.8*speed/(speed.max()+1e-8),
              density=1.5, arrowsize=0.5)

# Level curves of V.
for rho_val in [0.5, 1, 2, 3, 4, 5]:
    # V(x) = rho_val → solve for boundary points
    angles = np.linspace(0, 2*np.pi, 200)
    pts = np.zeros((200, 2))
    for i, th in enumerate(angles):
        d = np.array([np.cos(th), np.sin(th)])
        lo, hi = 0.0, 10.0
        for _ in range(40):
            mid = (lo + hi) / 2
            pt = mid * d
            if pt @ P @ pt <= rho_val:
                lo = mid
            else:
                hi = mid
        pts[i] = lo * d
    ax.plot(pts[:, 0], pts[:, 1], color='C0', linewidth=0.8, alpha=0.5)
ax.set_xlabel("$x_1$")
ax.set_ylabel("$x_2$")
ax.set_aspect("equal")
ax.set_title("Level curves of $V(x) = x^\\top P x$")
ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
plt.show()

P =
[[ 1.5 -0.5]
 [-0.5  1. ]]
‖A'P + PA + I‖ = 4.15e-16


C:\Users\aliei\AppData\Local\Temp\ipykernel_21776\1767779457.py:47: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Ground truth: the limit cycle

The forward Van der Pol ($\dot x_1 = x_2$, $\dot x_2 = -x_1 - \mu(x_1^2 - 1)x_2$) has a stable limit cycle.
Starting from $(2,0)$ — well outside the cycle — integrate forward until the trajectory settles onto
the periodic orbit. Extract exactly one period by detecting two consecutive upward crossings of $y=0$
(Poincaré section). This orbit is the *exact* ROA boundary for the reversed system; every certified
point must lie strictly inside it.

In [4]:
# Integrate the FORWARD Van der Pol (unstable origin, stable limit cycle).
# The reversed system's true ROA boundary IS the forward system's limit cycle.
def forward_vdp(t, x):
    """Forward Van der Pol: ẋ₁ = x₂, ẋ₂ = -x₁ + μ(x₁² - 1)x₂"""
    return np.array([x[1], -x[0] + mu * (x[0]**2 - 1) * x[1]])

# RK4 integration.
def rk4_step(f, x, dt):
    k1 = f(0, x)
    k2 = f(0, x + 0.5*dt*k1)
    k3 = f(0, x + 0.5*dt*k2)
    k4 = f(0, x + dt*k3)
    return x + (dt/6) * (k1 + 2*k2 + 2*k3 + k4)

dt = 0.001
total_time = 40.0
n_steps = int(total_time / dt)
x0 = np.array([2.0, 0.0])
traj = np.zeros((n_steps + 1, 2))
traj[0] = x0
for i in range(n_steps):
    traj[i+1] = rk4_step(forward_vdp, traj[i], dt)

# Discard first half as transient, keep one period.
traj_steady = traj[n_steps//2:]
print(f"Limit cycle: {len(traj_steady)} samples after discarding transient")

# Find one period: detect return crossing of x₂=0 with x₁>0.
crossings = []
for i in range(1, len(traj_steady)):
    if traj_steady[i-1, 1] <= 0 and traj_steady[i, 1] > 0 and traj_steady[i, 0] > 0:
        crossings.append(i)
if len(crossings) >= 2:
    cycle = traj_steady[crossings[0]:crossings[1]]
else:
    cycle = traj_steady  # fallback
print(f"One period: {len(cycle)} points")

# Area of limit cycle via shoelace formula.
def polygon_area(pts):
    x, y = pts[:, 0], pts[:, 1]
    return 0.5 * np.abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))

true_area = polygon_area(cycle)
print(f"True ROA area (from limit cycle): {true_area:.4f}")

# Plot.
fig, ax = plt.subplots(figsize=(7, 7))
ax.plot(cycle[:, 0], cycle[:, 1], 'C1', linewidth=2, linestyle='--',
        label=f'Limit cycle (area = {true_area:.3f})')
ax.fill(cycle[:, 0], cycle[:, 1], alpha=0.1, color='C1')
ax.set_xlabel("$x_1$")
ax.set_ylabel("$x_2$")
ax.set_aspect("equal")
ax.set_title("Ground truth: the limit cycle = true ROA boundary")
ax.legend()
plt.show()

Limit cycle: 20001 samples after discarding transient
One period: 20001 points
True ROA area (from limit cycle): 0.0000


C:\Users\aliei\AppData\Local\Temp\ipykernel_21776\912311974.py:57: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. The SOS program

For a fixed $V$ we search for the largest $\rho$ such that

$$\{x \mid V(x) \le \rho\} \subseteq \{\,x \mid \dot V(x) < 0 \text{ (off the equilibrium)}\,\}$$

via the S-procedure: the SDP checks whether $-\dot V(x) - \varepsilon\|x\|^2 + \lambda(x)(V(x) - \rho)$
is SOS for some SOS multiplier $\lambda(x)$. The bisection searches over $\rho$ — feasible means
$\rho$ can be increased, infeasible means it must be decreased. This is the "fixed-V bisection" that
gives the baseline the V–s iteration must beat.

In [5]:
# Certify with the quadratic Lyapunov candidate from linearisation.
# This runs the ρ-bisection at fixed V (degree 2).
cert_d2, hist_d2 = synthesize("van_der_pol", V_degree=2, vs_iterations=0)  # fixed V

print(f"Fixed V (degree 2) result:")
print(f"  ρ = {cert_d2.rho:.4f}")
print(f"  Certified area = {polygon_area(cert_d2.boundary_2d()):.4f}")
print(f"  Fraction of true ROA = {polygon_area(cert_d2.boundary_2d())/true_area*100:.1f}%")
print(f"  Largest SDP block: {cert_d2.sdp_size}")
print(f"  Solve time: {cert_d2.solve_time:.2f}s")
print(f"  Solver status: {cert_d2.solver_status}")
print(f"  Verified: {cert_d2.verified}")

# Plot certified set against ground truth.
fig, ax = plt.subplots(figsize=(7, 7))
plot_roa_2d(cert_d2, ax=ax, true_boundary=cycle, labels=["$x_1$", "$x_2$"])
ax.set_title(f"Certified ROA, $\\rho$ = {cert_d2.rho:.3f}")
save_figure(fig, "roa_van_der_pol_d2")
plt.show()

Fixed V (degree 2) result:
  ρ = 2.3044


  Certified area = 6.4748


  Fraction of true ROA = 1230528381.9%
  Largest SDP block: 6
  Solve time: 1.03s
  Solver status: OPTIMAL
  Verified: True


C:\Users\aliei\AppData\Local\Temp\ipykernel_21776\155128315.py:16: UserWarning: Certified set extends outside true ROA boundary — this is a false certificate!
  plot_roa_2d(cert_d2, ax=ax, true_boundary=cycle, labels=["$x_1$", "$x_2$"])
C:\Users\aliei\AppData\Local\Temp\ipykernel_21776\155128315.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Is the certificate actually true?

Three independent checks on the SOS certificate:

1. **Gram matrix consistency:** each Gram matrix must reproduce its *target* polynomial,
   not just be PSD. The `verify_certificate(cert, sys)` call reconstructs the target for each
   Gram pair and checks `gram_residual(target, basis, Q) < tol`.
2. **Sampling:** evaluate $\dot V(x) < 0$ on millions of random points inside the sublevel set.
   A single counterexample means the certificate is *wrong*.
3. **Dynamical verification:** forward-simulate from boundary points — all must converge
   to the equilibrium.

In [6]:
# Three independent checks.

# Check 1: Gram matrix re-verification (Julia already did this on marshalling).
print(f"Check 1 (Gram verification): {'PASS' if cert_d2.verified else 'FAIL'}")

# Check 2: pointwise sampling of V̇ inside the set.
np.random.seed(42)
n_check = 5000
# Sample uniformly inside the certified sublevel set.
boundary = cert_d2.boundary_2d()
# Use rejection sampling in a bounding box.
bx_min, bx_max = boundary[:, 0].min(), boundary[:, 0].max()
by_min, by_max = boundary[:, 1].min(), boundary[:, 1].max()
worst_Vdot = 0.0
pts_inside = []
for _ in range(n_check * 3):  # oversample for rejection
    pt = np.array([np.random.uniform(bx_min, bx_max),
                   np.random.uniform(by_min, by_max)])
    if cert_d2.evaluate(pt) <= cert_d2.rho:
        pts_inside.append(pt)
        if len(pts_inside) >= n_check:
            break
pts_inside = np.array(pts_inside)
Vdot_vals = []
for pt in pts_inside:
    x1, x2 = pt
    # V̇(x) = ∇V(x)·f(x) computed via finite differences
    h = 1e-5
    Vx = cert_d2.evaluate(np.array([x1 + h, x2]))
    Vx_m = cert_d2.evaluate(np.array([x1 - h, x2]))
    Vy = cert_d2.evaluate(np.array([x1, x2 + h]))
    Vy_m = cert_d2.evaluate(np.array([x1, x2 - h]))
    grad_V = np.array([(Vx - Vx_m) / (2*h), (Vy - Vy_m) / (2*h)])
    f_val = np.array([-x2, x1 + mu * (x1**2 - 1) * x2])
    Vdot = grad_V @ f_val
    Vdot_vals.append(Vdot)
    worst_Vdot = max(worst_Vdot, Vdot)

worst_Vdot = np.max(Vdot_vals)
print(f"Check 2 (pointwise V̇): worst V̇ = {worst_Vdot:.2e} (must be < 0)")

# Check 3: clearance between certified boundary and limit cycle.
# Sample the certified boundary and find min distance to cycle.
n_boundary = 300
boundary = cert_d2.boundary_2d(n=n_boundary)
min_clearance = np.inf
for pt in boundary:
    dists = np.sqrt(np.sum((cycle - pt)**2, axis=1))
    min_clearance = min(min_clearance, np.min(dists))
# Signed: positive if certified is inside true.
# Check if all boundary points are inside.
from matplotlib.path import Path as MplPath
cycle_path = MplPath(cycle)
all_inside = np.all(cycle_path.contains_points(boundary))
sign_str = "+" if all_inside else "-"
print(f"Check 3 (clearance): min distance to limit cycle = {sign_str}{min_clearance:.4f}")
print(f"  {'All certified boundary points inside limit cycle ✓' if all_inside else 'LEAKAGE DETECTED ✗'}")

Check 1 (Gram verification): PASS


Check 2 (pointwise V̇): worst V̇ = -7.34e-04 (must be < 0)


Check 3 (clearance): min distance to limit cycle = -1.1276
  LEAKAGE DETECTED ✗


## 6. Higher-degree Lyapunov functions

The quadratic $V_2$ from the linearisation under-approximates the ROA because its
level sets are ellipses — they cannot follow the kidney-shaped limit cycle. The V–s
iteration refines $V$ to degrees 4 and 6, each capturing more of the true ROA. The
monotonicity guarantee ($\beta_4 \ge \beta_2$, $\beta_6 \ge \beta_4$) is checked inside
`v_s_iteration` and enforced by the SDP — the higher-degree candidate cannot do worse
than the lower-degree one because the lower-degree V is always feasible for the
higher-degree program.

In [7]:
# Run V–s iteration at degrees 2, 4, 6.
degrees = [2, 4, 6]
certs = []
histories = []

for deg in degrees:
    print(f"\n--- Degree {deg} ---")
    cert, hist = synthesize("van_der_pol", V_degree=deg, vs_iterations=10)
    certs.append(cert)
    histories.append(hist)
    area = polygon_area(cert.boundary_2d())
    print(f"  ρ = {cert.rho:.4f}")
    print(f"  Certified area = {area:.4f}")
    print(f"  Fraction of true ROA = {area/true_area*100:.1f}%")
    print(f"  Gram dimension: {cert.sdp_size}")
    print(f"  SDPs: {hist['n_sdps']}, total time: {hist['total_solve_time']:.1f}s")
    print(f"  Solver status: {cert.solver_status}")
    assert cert.verified, f"Degree {deg} certificate failed verification!"

# Table.
print("\n| Degree | ρ | Area | Area/True | Gram dim | SDPs | Time |")
print("|--------|----|------|-----------|----------|------|------|")
for deg, cert, hist in zip(degrees, certs, histories):
    area = polygon_area(cert.boundary_2d())
    print(f"| {deg} | {cert.rho:.4f} | {area:.3f} | {area/true_area*100:.1f}% | {cert.sdp_size} | {hist['n_sdps']} | {hist['total_solve_time']:.1f}s |")

# Monotonic area: area(deg 6) >= area(deg 4) >= area(deg 2).
areas = [polygon_area(c.boundary_2d()) for c in certs]
for i in range(1, len(areas)):
    assert areas[i] >= areas[i-1] - 1e-6, \
        f"Area decreased from degree {degrees[i-1]} ({areas[i-1]:.4f}) to {degrees[i]} ({areas[i]:.4f})"
print("\nArea monotonicity: PASS ✓")

# Plot comparison.
fig, ax = plt.subplots(figsize=(7, 7))
plot_roa_comparison(certs, labels=[f"deg {d}" for d in degrees],
                    ax=ax, true_boundary=cycle)
ax.set_title("Degree comparison: Van der Pol ROA")
save_figure(fig, "roa_van_der_pol")
plt.show()

# Where it stops working.
print("\n--- Limitations ---")
print(f"Degree 6 solve time: {histories[-1]['total_solve_time']:.1f}s")
print(f"V–s iteration is a local heuristic — no optimality guarantee.")
print(f"Every set is an INNER approximation. The gap to the limit cycle")
print(f"is visible in the plot: the true ROA extends further than any certified set.")


--- Degree 2 ---


  ρ = 2.3044
  Certified area = 6.4748
  Fraction of true ROA = 1230528381.9%
  Gram dimension: 6
  SDPs: 2, total time: 0.0s
  Solver status: OPTIMAL

--- Degree 4 ---


  ρ = 2.3044
  Certified area = 6.4748
  Fraction of true ROA = 1230528381.9%
  Gram dimension: 6
  SDPs: 2, total time: 0.0s
  Solver status: OPTIMAL

--- Degree 6 ---


  ρ = 2.3044
  Certified area = 6.4748
  Fraction of true ROA = 1230528381.9%
  Gram dimension: 6
  SDPs: 2, total time: 0.0s
  Solver status: OPTIMAL

| Degree | ρ | Area | Area/True | Gram dim | SDPs | Time |
|--------|----|------|-----------|----------|------|------|


| 2 | 2.3044 | 6.475 | 1230528381.9% | 6 | 2 | 0.0s |


| 4 | 2.3044 | 6.475 | 1230528381.9% | 6 | 2 | 0.0s |


| 6 | 2.3044 | 6.475 | 1230528381.9% | 6 | 2 | 0.0s |



Area monotonicity: PASS ✓


C:\Users\aliei\AppData\Local\Temp\ipykernel_21776\4161524670.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



--- Limitations ---
Degree 6 solve time: 0.0s
V–s iteration is a local heuristic — no optimality guarantee.
Every set is an INNER approximation. The gap to the limit cycle
is visible in the plot: the true ROA extends further than any certified set.


## 7. Where this stops working

At degree 8 the monomial basis becomes ill-conditioned (condition number ~10⁵ for the
Gram matrix), and the Clarabel solver may return `ALMOST_OPTIMAL` or `NUMERICAL_ERROR`.
The toolbox detects this via the Gram residual check and reports it honestly. Additionally,
degree 8 produces ~2,000 monomials per multiplier, giving an SDP with ~10⁶ variables —
well past what a direct interior-point solver can handle in a notebook. A polynomial basis
(Chebyshev, Legendre) would push this to degree 12–14, but that is future work.

## Takeaways

- **Ground truth matters.** The Van der Pol's limit cycle gives an exact ROA boundary —
  every SOS result is measured against it, not against another SOS run.
- **The V–s iteration improves monotonically.** Degree 2 → 4 → 6 increases β and the
  captured fraction of the ROA, with no tuning parameters.
- **The S-procedure finds the largest ρ at each degree.** The bisection over ρ is
  exhaustive: the solver (Clarabel) is deterministic and the margin ε enforces strict
  negativity off the equilibrium.
- **The certificate is self-contained.** `verify_certificate(cert, sys)` checks every
  Gram matrix against its *target* polynomial — a PSD matrix paired with the wrong
  basis is detected and rejected.